# Stochastic Gradient Descent (SGD)

## Load Dataset from sklearn

In [1]:
import pandas as pd # 데이터 분석용 pandas 임포트
import seaborn as sns # 시각화 도구 seaborn 임포트
import matplotlib.pyplot as plt # 그래프 출력용 matplotlib 임포트

from sklearn.preprocessing import StandardScaler # 데이터 표준화 스케일러 임포트

In [2]:
from sklearn.datasets import fetch_california_housing # 캘리포니아 주택 데이터 로더 임포트
california = fetch_california_housing() # 데이터셋 로드

In [3]:
df = pd.DataFrame(california.data, columns=california.feature_names) # 특성 데이터를 데이터프레임으로 변환
df["Target"] = california.target # 정답(주택 가격) 컬럼 추가
df.tail() # 데이터 하단 출력

In [4]:
sns.pairplot(df.sample(1000)) # 1000개 샘플에 대한 상관관계 산점도 행렬 시각화
plt.show() # 그래프 표시

In [ ]:
scaler = StandardScaler() # 스케일러 객체 생성
scaler.fit(df.values[:, :-1]) # 입력 특성 데이터에 대해 평균/표준편차 계산
df.iloc[:, :-1] = scaler.transform(df.values[:, :-1]) # 입력 데이터 표준화 적용

In [6]:
print(df.values[:3,:5]) # 스케일링된 데이터 상위 일부 확인

In [7]:
sns.pairplot(df.sample(1000)) # 스케일링 후 데이터 상관관계 재확인
plt.show() # 그래프 표시

## Train Model with PyTorch

In [8]:
import torch # PyTorch 임포트
import torch.nn as nn # 신경망 모듈 임포트
import torch.nn.functional as F # 신경망 함수 모듈 임포트
import torch.optim as optim # 최적화 알고리즘 모듈 임포트

In [9]:
data = torch.from_numpy(df.values).float() # numpy 데이터를 PyTorch 실수형 텐서로 변환

data.shape # 전체 데이터 형상 확인

In [10]:
x = data[:, :-1] # 입력 특성 추출
y = data[:, -1:] # 정답 라벨 추출

print(x.shape, y.shape) # 입력/정답 데이터 형상 출력

In [11]:
n_epochs = 4000 # 학습 에포크 수 설정
batch_size = 256 # 미니배치 크기 설정
print_interval = 200 # 손실값 출력 간격 설정
learning_rate = 1e-2 # 학습률 설정

## Build Models

In [12]:
model = nn.Sequential( # 순차적 신경망 모델 정의
    nn.Linear(x.size(-1), 6), # 8차원 입력 -> 6개 노드 선형 레이어
    nn.LeakyReLU(), # LeakyReLU 활성화 함수
    nn.Linear(6, 5), # 6개 노드 -> 5개 노드 선형 레이어
    nn.LeakyReLU(), # LeakyReLU 활성화 함수
    nn.Linear(5, 4), # 5개 노드 -> 4개 노드 선형 레이어
    nn.LeakyReLU(), # LeakyReLU 활성화 함수
    nn.Linear(4, 3), # 4개 노드 -> 3개 노드 선형 레이어
    nn.LeakyReLU(), # LeakyReLU 활성화 함수
    nn.Linear(3, y.size(-1)), # 3개 노드 -> 1차원 출력 선형 레이어
)

model # 모델 구조 요약 출력

In [13]:
optimizer = optim.SGD(model.parameters(), lr=learning_rate) # SGD 최적화기 설정

In [14]:
for i in range(n_epochs): # 에포크 반복
    indices = torch.randperm(x.size(0)) # 무작위 인덱스 생성
    x_ = torch.index_select(x, dim=0, index=indices) # 데이터 셔플링
    y_ = torch.index_select(y, dim=0, index=indices) # 라벨 셔플링
    
    x_ = x_.split(batch_size, dim=0) # 데이터를 미니배치로 분할
    y_ = y_.split(batch_size, dim=0) # 라벨을 미니배치로 분할
    
    y_hat = [] # 예측값 저장용 리스트
    total_loss = 0 # 에포크별 총 손실 초기화
    
    for x_i, y_i in zip(x_, y_): # 미니배치 학습 루프
        y_hat_i = model(x_i) # 모델 예측
        loss = F.mse_loss(y_hat_i, y_i) # MSE 손실 계산

        optimizer.zero_grad() # 그래디언트 초기화
        loss.backward() # 역전파 수행

        optimizer.step() # 파라미터 업데이트
        
        total_loss += float(loss) # 손실 합산
        y_hat += [y_hat_i] # 예측값 수집

    total_loss = total_loss / len(x_) # 평균 손실 계산
    if (i + 1) % print_interval == 0: # 출력 간격마다 로그 표시
        print('Epoch %d: loss=%.4e' % (i + 1, total_loss))
    
y_hat = torch.cat(y_hat, dim=0) # 미니배치별 예측값 병합
y = torch.cat(y_, dim=0) # 미니배치별 실제값 병합

## Let's see the result!

In [16]:
df = pd.DataFrame(torch.cat([y, y_hat], dim=1).detach().numpy(), # 실제값과 예측값 결합 후 numpy 변환
                  columns=["y", "y_hat"]) # 컬럼명 지정

sns.pairplot(df, height=5) # 실제값 대비 예측값 분포 시각화
plt.show() # 그래프 표시